<a href="https://www.kaggle.com/code/hbad0612/heart-disease-prediction-with-basic-class-models?scriptVersionId=344064046" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/haiderrasoolqadri/heart-disease-dataset-uci/heart_disease_uci.csv')
df.head()

# Conduct EDA for the heart_disease_uci.csv dataset

In [ ]:
print(df.info())
print(df.shape)

In [ ]:
# Check for NaN values
df.isnull().sum()

In [ ]:
# Inspect missing values percentage for each variable
missing_percentage = (df.isnull().sum()/len(df))*100
print(missing_percentage)

In [ ]:
# As ca and thal columns have more than 50% missing values
# Drop ca and thal column in the dataset
df1 = df.drop(columns = ['ca', 'thal'])

In [ ]:
# Import library
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
print(df1.describe(include='all'))

In [ ]:
# Distribution of heart disease
df1['num_binary'] = (df1['num'] > 0).astype(int) # 0=no disease, 1=disease

In [ ]:
# Select features and target to prepare for training the model
# X features
X = df1.drop(columns=['id', 'dataset', 'num', 'num_binary'])

# y target
y = df1['num_binary']

print('X features:')
print(X.columns)
print("\nTarget distribution:")
print(y.value_counts())

In [ ]:
# Distribution of heart disease severity
plt.figure(figsize=(8, 5))
sns.countplot(x='num_binary', data=df1, hue='num_binary', legend=False)
plt.title('Distribution of Heart Disease Severity (0-4)')
plt.xlabel('Num (0=No Disease, 1-4=Severity)')
plt.ylabel('Count')
plt.show()

# Binary distribution
print(df1['num_binary'].value_counts(normalize=True) * 100)

In [ ]:
# Correlation heatmap for numerical features
# Select numerical features
numeric_features = X.select_dtypes(include=['int', 'float']).columns.tolist()

# Calculate correlation matrix
corr_matrix = X[numeric_features].corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5
)

plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

# Machine Learning Models
### Using basic classification models: Logistic Regression, Decision Tree, and Random Forest.

In [ ]:
# Train/test split 
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

In [ ]:
# Handle missing values + categorical variables
# Dataset contains both:
# Numerical variables: age, trestbps, chol, thalch, oldpeak
# Categorical variables: sex, cp, fbs, restecg, exang, slope
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Identify categorical column types
categorical_features = X.select_dtypes(include=['object', 'bool', 'category']).columns

print("Numeric features:")
print(list(numeric_features))

print("\nCategorical features:")
print(list(categorical_features))

In [ ]:
# Create preprocessing pipeline
# Numerical variables
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical variables
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

### Logistic Regression:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Use Logistic Regression pipeline as lr
lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Train
lr.fit(X_train, y_train)

# Predict
y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:, 1]

# print("Logistic Regression")
print("--------------------")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

### Decision Tree:

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(
        max_depth=4,
        random_state=42
    ))
])

dt.fit(X_train, y_train)

y_pred_tree = dt.predict(X_test)
y_prob_tree = dt.predict_proba(X_test)[:, 1]

print("Decision Tree")
print("--------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_tree))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_tree))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tree))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tree))

### Random Forest:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest")
print("--------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

## Compare all three models

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Decision Tree',
        'Random Forest'
    ],

    'Accuracy': [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_tree),
        accuracy_score(y_test, y_pred_rf)
    ],

    'Precision': [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_pred_tree),
        precision_score(y_test, y_pred_rf)
    ],

    'Recall': [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_pred_tree),
        recall_score(y_test, y_pred_rf)
    ],

    'F1 Score': [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_tree),
        f1_score(y_test, y_pred_rf)
    ],

    'ROC-AUC': [
        roc_auc_score(y_test, y_prob),
        roc_auc_score(y_test, y_prob_tree),
        roc_auc_score(y_test, y_prob_rf)
    ]
})

results.round(2)

#### Interpretation for the results:
- Random Forest is the strongest models across almost every metric. It detects about 92% of patients who actually have heart disease (Recall = 0.92). It has good separation between disease and non-disease cases (ROC-AUC = 0.92).
- Logistic Regression is the next competitive one. ROC-AUC of 0.90 and recall of 0.88 are quite good. Logistic Regression has the advantage of being easier to interpret clinically.
- Decision Tree is slightly weaker and is likely more sensitive to the specific train/test split.